In [5]:
"""
DeepShip WAV EDA Script
========================
Folder structure:
    /Users/zabir/Downloads/
        Cargo/
            20171104-1/
                1.wav
        Cargo 2/
        Cargo 3/
        Passengership/
        Passengership 2/
        Passengership 3/

Run:
    pip install librosa soundfile pandas matplotlib seaborn tqdm
    python deepship_eda.py
"""

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
from tqdm import tqdm
from pathlib import Path

# ─────────────────────────────────────────────
ROOT_DIR   = "/Users/zabir/Downloads"
OUTPUT_DIR = "/Users/zabir/Downloads/deepship_eda_output"
# ─────────────────────────────────────────────

Path(OUTPUT_DIR).mkdir(exist_ok=True)
sns.set_theme(style="darkgrid", palette="tab10")


def get_label(class_folder_name: str) -> str:
    """'Cargo 2' -> 'Cargo',  'Passengership 3' -> 'Passengership'"""
    return re.sub(r'\s+\d+$', '', class_folder_name).strip()


def collect_metadata(root: str) -> pd.DataFrame:
    root_path = Path(root).resolve()
    wav_files = list(root_path.rglob("*.wav"))
    print(f"Found {len(wav_files)} WAV files. Collecting metadata...")

    records = []
    for fp in tqdm(wav_files):
        try:
            # top-level subfolder relative to root
            rel_parts = fp.relative_to(root_path).parts
            class_folder = rel_parts[0]

            # skip anything that isn't a known class folder
            if not class_folder.startswith(("Cargo", "Passengership")):
                continue

            info    = sf.info(str(fp))
            size_mb = fp.stat().st_size / 1e6

            records.append({
                "filepath":     str(fp),
                "filename":     fp.name,
                "class_folder": class_folder,
                "label":        get_label(class_folder),
                "session":      fp.parent.name,
                "duration_s":   round(info.duration, 2),
                "sample_rate":  info.samplerate,
                "channels":     info.channels,
                "frames":       info.frames,
                "subtype":      info.subtype,
                "size_mb":      round(size_mb, 3),
            })
        except Exception as e:
            print(f"  [WARN] Skipped {fp}: {e}")

    df = pd.DataFrame(records)
    print(f"Metadata collected: {len(df)} files, {df['size_mb'].sum():.1f} MB total")
    return df


def print_summary(df: pd.DataFrame):
    print("\n" + "="*60)
    print("DATASET SUMMARY")
    print("="*60)
    print(f"Total files     : {len(df)}")
    print(f"Total size      : {df['size_mb'].sum():.1f} MB  ({df['size_mb'].sum()/1024:.2f} GB)")
    print(f"Labels          : {sorted(df['label'].unique())}")
    print(f"Class folders   : {sorted(df['class_folder'].unique())}")
    print(f"Sample rates    : {df['sample_rate'].value_counts().to_dict()}")
    print(f"Channels        : {df['channels'].value_counts().to_dict()}")
    print(f"Bit formats     : {df['subtype'].value_counts().to_dict()}")
    print(f"\nDuration (seconds):")
    print(df['duration_s'].describe().round(2).to_string())
    print(f"\nPer-label breakdown:")
    print(df.groupby('label').agg(
        files         = ('filename',   'count'),
        sessions      = ('session',    'nunique'),
        total_dur_min = ('duration_s', lambda x: round(x.sum() / 60, 1)),
        mean_dur_s    = ('duration_s', lambda x: round(x.mean(), 1)),
        min_dur_s     = ('duration_s', 'min'),
        max_dur_s     = ('duration_s', 'max'),
        total_size_mb = ('size_mb',    lambda x: round(x.sum(), 1)),
    ).to_string())


def plot_class_distribution(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    palette = sns.color_palette("tab10", df['label'].nunique())

    counts = df.groupby('label')['filename'].count().sort_values(ascending=False)
    axes[0].bar(counts.index, counts.values, color=palette)
    axes[0].set_title("File Count per Class")
    axes[0].set_xlabel("Class"); axes[0].set_ylabel("Number of Files")
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 0.2, str(v), ha='center', fontsize=10)

    dur = (df.groupby('label')['duration_s'].sum() / 60).sort_values(ascending=False)
    axes[1].bar(dur.index, dur.values, color=palette)
    axes[1].set_title("Total Audio Duration per Class (minutes)")
    axes[1].set_xlabel("Class"); axes[1].set_ylabel("Minutes")
    for i, v in enumerate(dur.values):
        axes[1].text(i, v + 0.5, f"{v:.0f}", ha='center', fontsize=10)

    plt.suptitle("Class Distribution", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/01_class_distribution.png", dpi=150)
    plt.close()
    print("  Saved: 01_class_distribution.png")


def plot_duration_distributions(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(df['duration_s'], bins=40, color='steelblue', edgecolor='white')
    axes[0].axvline(df['duration_s'].median(), color='red', linestyle='--',
                    label=f"Median: {df['duration_s'].median():.0f}s")
    axes[0].set_title("Overall File Duration Distribution")
    axes[0].set_xlabel("Duration (seconds)"); axes[0].set_ylabel("Count")
    axes[0].legend()

    sns.boxplot(data=df, x='label', y='duration_s', ax=axes[1],
                palette="tab10", order=sorted(df['label'].unique()))
    axes[1].set_title("Duration by Class")
    axes[1].set_xlabel("Class"); axes[1].set_ylabel("Duration (seconds)")

    plt.suptitle("Audio Duration Analysis", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/02_duration_distributions.png", dpi=150)
    plt.close()
    print("  Saved: 02_duration_distributions.png")


def plot_file_size(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(df['size_mb'], bins=40, color='coral', edgecolor='white')
    axes[0].set_title("File Size Distribution")
    axes[0].set_xlabel("Size (MB)"); axes[0].set_ylabel("Count")

    size_by_class = df.groupby('label')['size_mb'].sum()
    axes[1].pie(size_by_class, labels=size_by_class.index, autopct='%1.1f%%',
                colors=sns.color_palette("tab10", len(size_by_class)))
    axes[1].set_title("Storage Share by Class")

    plt.suptitle("File Size Analysis", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/03_file_size.png", dpi=150)
    plt.close()
    print("  Saved: 03_file_size.png")


def plot_waveforms_and_spectrograms(df: pd.DataFrame):
    classes = sorted(df['label'].unique())
    print(f"\n  Generating waveforms & spectrograms ({len(classes)} classes)...")

    for label in classes:
        row = df[df['label'] == label].iloc[0]
        try:
            y, sr = librosa.load(row['filepath'], sr=None, duration=30)

            fig, axes = plt.subplots(2, 1, figsize=(12, 6))

            librosa.display.waveshow(y, sr=sr, ax=axes[0], color='steelblue')
            axes[0].set_title(f"Waveform — {label}  |  {row['session']}  |  sr={sr} Hz")
            axes[0].set_xlabel("Time (s)")

            S    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=sr // 2)
            S_db = librosa.power_to_db(S, ref=np.max)
            img  = librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel',
                                            fmax=sr // 2, ax=axes[1], cmap='magma')
            fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
            axes[1].set_title(f"Mel Spectrogram — {label}")

            plt.tight_layout()
            out = f"{OUTPUT_DIR}/viz_{label.replace(' ', '_')}.png"
            plt.savefig(out, dpi=150)
            plt.close()
            print(f"    Saved: viz_{label}.png")

        except Exception as e:
            print(f"    [WARN] Could not visualize {label}: {e}")


def compute_and_plot_mfcc(df: pd.DataFrame):
    print("\n  Computing MFCCs (30s cap per file)...")
    N_MFCC  = 20
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            y, sr = librosa.load(row['filepath'], sr=None, duration=30)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
            rec   = {"label": row['label']}
            for i, v in enumerate(mfccs.mean(axis=1)):
                rec[f"mfcc_{i+1}"] = v
            records.append(rec)
        except Exception:
            pass

    mfcc_df = pd.DataFrame(records)
    mfcc_df.to_csv(f"{OUTPUT_DIR}/mfcc_stats.csv", index=False)

    # Heatmap
    mfcc_cols      = [c for c in mfcc_df.columns if c.startswith("mfcc_")]
    mean_by_class  = mfcc_df.groupby('label')[mfcc_cols].mean()
    plt.figure(figsize=(14, 4))
    sns.heatmap(mean_by_class, annot=False, cmap='coolwarm', linewidths=0.5)
    plt.title("Mean MFCC Coefficients by Class")
    plt.xlabel("MFCC Coefficient")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/04_mfcc_heatmap.png", dpi=150)
    plt.close()
    print("  Saved: 04_mfcc_heatmap.png")

    # Violin — first 4 coefficients
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()
    for i, coef in enumerate([1, 2, 3, 4]):
        sns.violinplot(data=mfcc_df, x='label', y=f"mfcc_{coef}",
                       ax=axes[i], palette="tab10", inner="box",
                       order=sorted(mfcc_df['label'].unique()))
        axes[i].set_title(f"MFCC {coef} by Class")
        axes[i].set_xlabel("")
        axes[i].tick_params(axis='x', rotation=20)
    plt.suptitle("MFCC Distributions by Class (first 4 coefficients)", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/05_mfcc_violin.png", dpi=150)
    plt.close()
    print("  Saved: 05_mfcc_violin.png")

    return mfcc_df


if __name__ == "__main__":
    df = collect_metadata(ROOT_DIR)

    if df.empty:
        print("ERROR: No WAV files found. Check ROOT_DIR path.")
    else:
        df.to_csv(f"{OUTPUT_DIR}/metadata.csv", index=False)
        print(f"Metadata saved -> {OUTPUT_DIR}/metadata.csv")

        print_summary(df)

        print("\nGenerating plots...")
        plot_class_distribution(df)
        plot_duration_distributions(df)
        plot_file_size(df)
        plot_waveforms_and_spectrograms(df)

        # Comment out the line below if you want quick results (MFCC is slowest)
        mfcc_df = compute_and_plot_mfcc(df)

        print(f"\nDone! All outputs saved to: {OUTPUT_DIR}/")

Found 300 WAV files. Collecting metadata...


100%|██████████| 300/300 [00:00<00:00, 6218.11it/s]

Metadata collected: 300 files, 10830.8 MB total
Metadata saved -> /Users/zabir/Downloads/deepship_eda_output/metadata.csv

DATASET SUMMARY
Total files     : 300
Total size      : 10830.8 MB  (10.58 GB)
Labels          : ['Cargo', 'Passengership']
Class folders   : ['Cargo', 'Cargo 2', 'Cargo 3', 'Passengership', 'Passengership 2', 'Passengership 3']
Sample rates    : {32000: 300}
Channels        : {1: 300}
Bit formats     : {'FLOAT': 296, 'PCM_24': 4}

Duration (seconds):
count     300.00
mean      283.05
std       216.41
min         6.00
25%       137.50
50%       279.50
75%       370.00
max      1887.00

Per-label breakdown:
               files  sessions  total_dur_min  mean_dur_s  min_dur_s  max_dur_s  total_size_mb
label                                                                                         
Cargo            109       109          641.6       353.2      183.0     1020.0         4927.4
Passengership    191       191          773.7       243.0        6.0     1887.0 

  Saved: 01_class_distribution.png
  Saved: 02_duration_distributions.png
  Saved: 03_file_size.png

  Generating waveforms & spectrograms (2 classes)...
    Saved: viz_Cargo.png
    Saved: viz_Passengership.png

  Computing MFCCs (30s cap per file)...


100%|██████████| 300/300 [00:07<00:00, 42.18it/s]


  Saved: 04_mfcc_heatmap.png
  Saved: 05_mfcc_violin.png

Done! All outputs saved to: /Users/zabir/Downloads/deepship_eda_output/
